In [1]:
!mkdir -p ~/work/transformer_chatbot/data/
!wget https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv -P ~/work/transformer_chatbot/data/

--2026-04-17 05:09:14--  https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv [following]
--2026-04-17 05:09:14--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘/home/jovyan/work/transformer_chatbot/data/ChatbotData.csv’

ChatbotData.csv     100%[===================>] 868.99K  5.64MB/s    in 0.2s    

2026-04-17 05:09:15 (5.64 MB/s) - ‘/home/jovyan/work/transformer_chatbot/da

In [5]:
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.0 MB/s eta 0:00:00 0:00:01


In [5]:
!pip install torch torchtext sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.9 MB/s eta 0:00:00 0:00:01


In [1]:
import pandas as pd
import os

# 파일 경로 설정
path_to_chatbot_data = os.getenv('HOME') + '/work/transformer_chatbot/data/ChatbotData.csv'
data = pd.read_csv(path_to_chatbot_data)

print(f"전체 데이터 개수: {len(data)}")
data.head()

전체 데이터 개수: 11823


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


# 한국어 전처리

In [2]:
import re

def preprocess_sentence(sentence):
    # 단어와 구두점 사이 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    # 여러 개의 공백을 하나로 합치기
    sentence = re.sub(r'[" "]+', " ", sentence)
    # 한글, 영문, 숫자, 구두점 외 제거
    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,]+", " ", sentence)
    sentence = sentence.strip()
    return sentence

data['Q'] = data['Q'].apply(preprocess_sentence)
data['A'] = data['A'].apply(preprocess_sentence)

# SentencePiece 모델 학습

In [3]:
import sentencepiece as spm

# 학습용 텍스트 파일 만들기
with open('chatbot.txt', 'w', encoding='utf-8') as f:
    for row in data['Q']: f.write(row + '\n')
    for row in data['A']: f.write(row + '\n')

# SentencePiece 학습 (VOCAB_SIZE적절히 설정)
VOCAB_SIZE = 8000
spm.SentencePieceTrainer.Train(
    '--input=chatbot.txt --model_prefix=korean_spm '
    f'--vocab_size={VOCAB_SIZE} --model_type=unigram '
    '--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3'
)

# 학습된 모델 로드
sp = spm.SentencePieceProcessor()
sp.Load('korean_spm.model')

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=chatbot.txt --model_prefix=korean_spm --vocab_size=8000 --model_type=unigram --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: chatbot.txt
  input_format: 
  model_prefix: korean_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_voc

True

# 토큰화 / 패딩

In [7]:
import numpy as np

MAX_LENGTH = 40

def tokenize_and_filter(inputs, outputs):
    tokenized_inputs, tokenized_outputs = [], []
    
    for (sentence1, sentence2) in zip(inputs, outputs):
        # SentencePiece(sp)를 사용하여 인코딩
        # <BOS> 토큰 + 문장 + <EOS> 토큰
        sentence1 = [sp.bos_id()] + sp.EncodeAsIds(sentence1) + [sp.eos_id()]
        sentence2 = [sp.bos_id()] + sp.EncodeAsIds(sentence2) + [sp.eos_id()]
        
        # 최대 길이 제한 내에 있는 것만 추가
        if len(sentence1) <= MAX_LENGTH and len(sentence2) <= MAX_LENGTH:
            tokenized_inputs.append(sentence1)
            tokenized_outputs.append(sentence2)
            
    # 파이토치는 패딩 처리를 위해 pad_sequence를 쓰거나 
    # numpy를 거쳐 torch.tensor로 변환하는 것이 편함.
    # 여기서는 동일한 길이를 위해 0(pad_id)으로 패딩을 채운다.
    def pad_sequences(sequences, maxlen, value=0):
        return [seq + [value] * (maxlen - len(seq)) for seq in sequences]

    tokenized_inputs = pad_sequences(tokenized_inputs, MAX_LENGTH)
    tokenized_outputs = pad_sequences(tokenized_outputs, MAX_LENGTH)
    
    return tokenized_inputs, tokenized_outputs

# data['Q']와 data['A'] 위에서 정의.
questions, answers = tokenize_and_filter(data['Q'], data['A'])

print(f"필터링 후 데이터 개수: {len(questions)}")

필터링 후 데이터 개수: 11823


In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. 파이토치용 커스텀 데이터셋 정의
class ChatbotDataset(Dataset):
    def __init__(self, questions, answers):
        self.questions = torch.tensor(questions, dtype=torch.long)
        self.answers = torch.tensor(answers, dtype=torch.long)

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, i):
        return self.questions[i], self.answers[i]
# 위에 만든 정수 시퀀스를 사용하여 로더 생성
dataset = ChatbotDataset(questions, answers)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [11]:
# 2. 트랜스포머 핵심 레이어: Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        
        assert d_model % num_heads == 0
        self.depth = d_model // num_heads
        
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)
        
    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)

    def forward(self, v, k, q, mask):
        batch_size = q.size(0)
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)
        
        # Scaled Dot-Product Attention (여기서 스케일링이 일어
        matmul_qk = torch.matmul(q, k.transpose(-1, -2))
        dk = torch.tensor(self.depth, dtype=torch.float32)
        scaled_attention_logits = matmul_qk / torch.sqrt(dk)
        
        if mask is not None:
            scaled_attention_logits += (mask * -1e9)
            
        attention_weights = torch.softmax(scaled_attention_logits, dim=-1)
        output = torch.matmul(attention_weights, v)
        output = output.permute(0, 2, 1, 3).contiguous()
        output = output.view(batch_size, -1, self.d_model)
        return self.dense(output)

# 모델 평가용 예측 함수 (Inference)

In [31]:
def predict(sentence, beam_size=3):
    model.eval()
    sentence = preprocess_sentence(sentence)
    
    input_ids = [sp.bos_id()] + sp.EncodeAsIds(sentence) + [sp.eos_id()]
    inputs = torch.tensor([input_ids]).to(device)

    # (확률 합계, 현재까지 생성된 토큰 리스트)
    beams = [(0, [sp.bos_id()])]

    for i in range(MAX_LENGTH):
        new_beams = []
        for prob_sum, tokens in beams:
            # 마지막 토큰이 EOS면 더 이상 생성하지 않고 유지
            if tokens[-1] == sp.eos_id():
                new_beams.append((prob_sum, tokens))
                continue
            
            curr_output = torch.tensor([tokens]).to(device)
            with torch.no_grad():
                predictions = model(inputs, curr_output)
            
            # 마지막 타임스텝의 확률 분포 추출
            log_probs = torch.log_softmax(predictions[:, -1, :], dim=-1)
            
            # 상위 beam_size개의 다음 토큰 후보 추출
            top_log_probs, top_ids = torch.topk(log_probs, beam_size)
            
            for j in range(beam_size):
                next_prob = top_log_probs[0][j].item()
                next_id = top_ids[0][j].item()
                new_beams.append((prob_sum + next_prob, tokens + [next_id]))
        
        # 모든 후보 중 확률 합이 높은 상위 beam_size개만 남김
        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]
        
        # 모든 후보가 EOS로 끝났다면 조기 종료
        if all(tokens[-1] == sp.eos_id() for _, tokens in beams):
            break

    # 가장 확률이 높은 최종 문장 선택
    best_tokens = beams[0][1]
    return sp.DecodeIds(best_tokens)

# 전체 트랜스포머 모델

In [32]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, units, dropout):
        super(Transformer, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        # 파이토치 표준 Transformer 레이어 사용
        self.transformer = nn.Transformer(
            d_model=d_model, 
            nhead=num_heads, 
            num_encoder_layers=num_layers, #인코더 층
            num_decoder_layers=num_layers, #디코더 
            dim_feedforward=units, 
            dropout=dropout,
            batch_first=True
        )
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # 마스크 생성 (Padding 및 Look-ahead)
        src = self.embedding(src) * np.sqrt(self.d_model)
        tgt = self.embedding(tgt) * np.sqrt(self.d_model)
        
        # 실제 트랜스포머 연산 (병렬 데이터 구축)
        output = self.transformer(src, tgt, tgt_mask=tgt_mask)
        return self.out(output)

# 모델 인스턴스화
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(
    vocab_size=8000, d_model=256, num_heads=8, num_layers=2, units=512, dropout=0.1
).to(device)

손실 함수와 최적화 도구 설정

In [33]:
criterion = nn.CrossEntropyLoss(ignore_index=0) # 패딩 토큰은 손실 계산 제외
optimizer = optim.Adam(model.parameters(), lr=1e-4)

학습 루프

In [17]:
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        questions, answers = batch
        questions, answers = questions.to(device), answers.to(device)
        
        # 디코더 입력: <BOS> I AM A STUDENT (마지막 토큰 제외)
        # 레이블: I AM A STUDENT <EOS> (첫 토큰 제외)
        dec_input = answers[:, :-1]
        labels = answers[:, 1:]
        
        # 타겟 마스크 생성 (미래 토큰을 보지 못하게 함)
        tgt_mask = model.transformer.generate_square_subsequent_mask(dec_input.size(1)).to(device)

        optimizer.zero_grad()
        outputs = model(questions, dec_input, tgt_mask=tgt_mask)
        
        loss = criterion(outputs.reshape(-1, 8000), labels.reshape(-1))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/20, Loss: 6.3047
Epoch 2/20, Loss: 5.2667
Epoch 3/20, Loss: 4.9863
Epoch 4/20, Loss: 4.7515
Epoch 5/20, Loss: 4.5489
Epoch 6/20, Loss: 4.3703
Epoch 7/20, Loss: 4.2076
Epoch 8/20, Loss: 4.0633
Epoch 9/20, Loss: 3.9288
Epoch 10/20, Loss: 3.7985
Epoch 11/20, Loss: 3.6756
Epoch 12/20, Loss: 3.5592
Epoch 13/20, Loss: 3.4458
Epoch 14/20, Loss: 3.3337
Epoch 15/20, Loss: 3.2250
Epoch 16/20, Loss: 3.1205
Epoch 17/20, Loss: 3.0182
Epoch 18/20, Loss: 2.9175
Epoch 19/20, Loss: 2.8179
Epoch 20/20, Loss: 2.7215


테스트

In [22]:
# 테스트 문장들
questions_test = [
    "안녕?",
    "오늘 기분이 어때?",
    "배고프다",
    "너는 누구니?",
    "고민이 있어"
]

print("="*30)
for q in questions_test:
    answer = predict(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 30)

Q: 안녕?
A: 꼭보네문이 실례 들었는데보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네이너보네
------------------------------
Q: 오늘 기분이 어때?
A: 착각하 꼰대 잘생겨 씨앗 주변 한편 좋은느 청소 확신 하셔 내려놓기 기본 조심해야 그대로찾 폭식증 주 나갈 우정 합 씨앗침 일주일이네 특 그대로 정상이 기본 한달반 했던 적절 무섭 썸타는넣침응 더럽 허무해 내려놓기콘
------------------------------
Q: 배고프다
A: 응 달에 무시하는 피시방 행복하 피시방 기원하우스 대답 축하해요 중요 화 그대로 나을까렸어 시원 들어줄문 열 알뜰 환승 몸무게감기 그냥일이 둘감기 싶은 의욕 키워분 산 조건 열 심심하다우선이 인연이 그대로 나을까렸어
------------------------------
Q: 너는 누구니?
A: 착각하 꼰대인건가시간이 투자 기름값 봤나 잘생겨 여러명 해야이너년째디달반 다음 둘보여 표현해 둘보여 둘보여 둘보여 둘보여 둘보여 둘보여 둘보여 둘보여 둘보여 둘보여 둘보여
------------------------------
Q: 고민이 있어
A: 떨쳐이너잡히네값달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반달반
------------------------------


In [25]:
# 1. 손실 함수와 최적화 도구 재설정 (패딩 토큰 index=0 제외)
criterion = nn.CrossEntropyLoss(ignore_index=0) 
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 2. 학습 루프 실행
EPOCHS = 200  # 에폭 200으로 상향

print("학습시작. 이번에는 챗봇이 제대로 말을 배워야한다")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        # 데이터 로드 및 GPU 이동
        questions_batch, answers_batch = batch
        questions_batch = questions_batch.to(device)
        answers_batch = answers_batch.to(device)
        
        # Teacher Forcing 입력과 레이블 분리
        # 입력: <BOS> 문장... (마지막 토큰 제외)
        # 레이블: 문장... <EOS> (첫 토큰 제외)
        dec_input = answers_batch[:, :-1]
        labels = answers_batch[:, 1:]
        
        # 디코더의 인과적 마스크(Look-ahead mask) 생성
        # 현재 토큰이 미래 토큰을 보지 못하게 차단
        tgt_mask = model.transformer.generate_square_subsequent_mask(dec_input.size(1)).to(device)

        optimizer.zero_grad()
        
        # 모델 예측
        outputs = model(questions_batch, dec_input, tgt_mask=tgt_mask)
        
        # Loss 계산 (reshape를 통해 차원 정렬: [배치*길이, 단어수])
        # ignore_index=0에 의해 패딩 부분의 손실은 계산되지 않습니다.
        loss = criterion(outputs.reshape(-1, 8000), labels.reshape(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    # 10에폭마다 중간 결과 출력 (진행 상황 확인용)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

print("학습 완료. 다시 predict 함수로 대화해보자")

학습시작. 이번에는 챗봇이 제대로 말을 배워야한다
Epoch 1/200, Loss: 0.6729
Epoch 5/200, Loss: 0.5298
Epoch 10/200, Loss: 0.3972
Epoch 15/200, Loss: 0.2990
Epoch 20/200, Loss: 0.2335
Epoch 25/200, Loss: 0.1878
Epoch 30/200, Loss: 0.1523
Epoch 35/200, Loss: 0.1282
Epoch 40/200, Loss: 0.1117
Epoch 45/200, Loss: 0.0998
Epoch 50/200, Loss: 0.0867
Epoch 55/200, Loss: 0.0802
Epoch 60/200, Loss: 0.0721
Epoch 65/200, Loss: 0.0670
Epoch 70/200, Loss: 0.0630
Epoch 75/200, Loss: 0.0579
Epoch 80/200, Loss: 0.0535
Epoch 85/200, Loss: 0.0526
Epoch 90/200, Loss: 0.0486
Epoch 95/200, Loss: 0.0473
Epoch 100/200, Loss: 0.0442
Epoch 105/200, Loss: 0.0441
Epoch 110/200, Loss: 0.0412
Epoch 115/200, Loss: 0.0385
Epoch 120/200, Loss: 0.0386
Epoch 125/200, Loss: 0.0362
Epoch 130/200, Loss: 0.0369
Epoch 135/200, Loss: 0.0350
Epoch 140/200, Loss: 0.0337
Epoch 145/200, Loss: 0.0304
Epoch 150/200, Loss: 0.0313
Epoch 155/200, Loss: 0.0300
Epoch 160/200, Loss: 0.0300
Epoch 165/200, Loss: 0.0311
Epoch 170/200, Loss: 0.0271
Epoch 175/200,

In [24]:
# 테스트 문장들 (100 에폭 돌린 결과)
questions_test = [
    "안녕?",
    "오늘 기분이 어때?",
    "배고프다",
    "너는 누구니?",
    "고민이 있어"
]

print("="*30)
for q in questions_test:
    answer = predict(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 30)

Q: 안녕?
A: 이제  ⁇  마세요 .
------------------------------
Q: 오늘 기분이 어때?
A: 그 분이 알 길죠 .
------------------------------
Q: 배고프다
A: 그 사람 마다할 수 있는 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그 사람 그
------------------------------
Q: 너는 누구니?
A: 보이는 옷이요 ?
------------------------------
Q: 고민이 있어
A: 그 사랑했던 사람을 위해서 .
------------------------------


In [26]:
# 테스트 문장들 (다시 200 에폭 돌린 결과)
questions_test = [
    "안녕?",
    "오늘 기분이 어때?",
    "배고프다",
    "너는 누구니?",
    "고민이 있어"
]

print("="*30)
for q in questions_test:
    answer = predict(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 30)

Q: 안녕?
A: 심경 변화 가능성 만들어 당신 .
------------------------------
Q: 오늘 기분이 어때?
A: 그 감정에 솔직한 아무는 데를 마음 같지 않을 때 사랑에 나이 그럴 확률이 없 없 없죠 .
------------------------------
Q: 배고프다
A: 그 분이 좋아하는 사람을 위해 에너지를 해요 .
------------------------------
Q: 너는 누구니?
A: 저는 위로해드리는 외로운 시간 그 누구 , 저요 .
------------------------------
Q: 고민이 있어
A: 그 다른 걸 발전 해보세요 .
------------------------------


개선된 Beam Search 버전의 predict 함수 적용 후

In [34]:
# 1. 손실 함수와 최적화 도구 재설정 (패딩 토큰 index=0 제외)
criterion = nn.CrossEntropyLoss(ignore_index=0) 
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 2. 학습 루프 실행
EPOCHS = 100  

print("학습시작. 이번에는 챗봇이 제대로 말을 배워야한다")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        # 데이터 로드 및 GPU 이동
        questions_batch, answers_batch = batch
        questions_batch = questions_batch.to(device)
        answers_batch = answers_batch.to(device)
        
        # Teacher Forcing 입력과 레이블 분리
        # 입력: <BOS> 문장... (마지막 토큰 제외)
        # 레이블: 문장... <EOS> (첫 토큰 제외)
        dec_input = answers_batch[:, :-1]
        labels = answers_batch[:, 1:]
        
        # 디코더의 Look-ahead mask 생성
        # 현재 토큰이 미래 토큰을 보지 못하게 차단
        tgt_mask = model.transformer.generate_square_subsequent_mask(dec_input.size(1)).to(device)

        optimizer.zero_grad()
        
        # 모델 예측
        outputs = model(questions_batch, dec_input, tgt_mask=tgt_mask)
        
        # Loss 계산 (reshape를 통해 차원 정렬: [배치*길이, 단어수])
        # ignore_index=0에 의해 패딩 부분의 손실은 계산되지 않음.
        loss = criterion(outputs.reshape(-1, 8000), labels.reshape(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    # 10에폭마다 중간 결과 출력 (진행 상황 확인용)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

print("학습 완료. 다시 predict 함수로 대화해보자")

학습시작. 이번에는 챗봇이 제대로 말을 배워야한다
Epoch 1/100, Loss: 6.3205
Epoch 5/100, Loss: 4.5390
Epoch 10/100, Loss: 3.7821
Epoch 15/100, Loss: 3.1880
Epoch 20/100, Loss: 2.6744
Epoch 25/100, Loss: 2.2112
Epoch 30/100, Loss: 1.7970
Epoch 35/100, Loss: 1.4442
Epoch 40/100, Loss: 1.1449
Epoch 45/100, Loss: 0.8879
Epoch 50/100, Loss: 0.6763
Epoch 55/100, Loss: 0.5179
Epoch 60/100, Loss: 0.3893
Epoch 65/100, Loss: 0.3046
Epoch 70/100, Loss: 0.2316
Epoch 75/100, Loss: 0.1877
Epoch 80/100, Loss: 0.1515
Epoch 85/100, Loss: 0.1322
Epoch 90/100, Loss: 0.1144
Epoch 95/100, Loss: 0.0984
Epoch 100/100, Loss: 0.0887
학습 완료. 다시 predict 함수로 대화해보자


In [35]:
# 테스트 문장들 (함수 수정 후 100 에폭 돌린 결과)
questions_test = [
    "안녕?",
    "오늘 기분이 어때?",
    "배고프다",
    "너는 누구니?",
    "고민이 있어"
]

print("="*30)
for q in questions_test:
    answer = predict(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 30)

Q: 안녕?
A: 안녕하세요 .
------------------------------
Q: 오늘 기분이 어때?
A: 모두 맞춰주가 있었나봐요 .
------------------------------
Q: 배고프다
A: 여름 언제나 일기예대한 두려움했겠어요 .
------------------------------
Q: 너는 누구니?
A: 하나씩 잊어가는 그 자체만으로 나을 수도 있어요 .
------------------------------
Q: 고민이 있어
A: 스타일에 잠 못 드는 밤인가봐요 .
------------------------------


# 트랜스포머 기반 한국어 챗봇 만들기

## 1. 프로젝트 개요
이번 프로젝트에서 트랜스포머 아키텍처를 사용하여 한국어 일상 대화 및 고민 상담이 가능한 챗봇을 구현해봤다. PyTorch 프레임워크를 기반으로 형태소 분석기 대신 Google의 SentencePiece를 사용하여 서브워드 토큰화를 진행했다.

---

## 2. 주요 수행 단계
### Step 1. 데이터 수집 및 전처리
- **데이터셋:** 송영숙 님의 한국어 챗봇 데이터(11,823개 샘플) 사용
- **전처리:** 정규표현식을 활용하여 특수문자와 구두점 양옆에 공백을 추가하고, 불필요한 공백과 노이즈를 제거하는 `preprocess_sentence` 함수 적용

### Step 2. 토큰화 (SentencePiece)
- **방식:** SubwordTextEncoder 대신 SentencePiece(BPE)를 사용하여 한국어의 교착어적 특성을 반영했음.
- **설정:** 8,000개의 단어 사전(Vocab Size)을 구축하고, `<BOS>`, `<EOS>`, `<PAD>`, `<UNK>` 토큰을 정의하여 학습의 안정성을 높임

### Step 3. 모델 설계 (PyTorch)
- **아키텍처:** Transformer (Multi-Head Attention, Positional Encoding 등 구현)
- **하이퍼파라미터:**
  - `d_model`: 256
  - `num_layers`: 2
  - `num_heads`: 8
  - `units`: 512
  - `dropout`: 0.1

### Step 4. 학습 및 평가
- **Loss Function:** 패딩 토큰을 무시하는 `CrossEntropyLoss(ignore_index=0)` 사용
- **Optimizer:** Adam
- **Epochs:** 100회 학습을 통해 Loss를 안정적으로 수렴시킴
- **Decoding:** Greedy Search의 단점인 문장 반복 현상을 해결하기 위해 **Beam Search(beam_size=3)** 기법 도입

---

## 3. 모델 결과 (Test)
학습이 완료된 모델에 5가지 대표 문장을 입력한 결과입니다.

| 질문 (Input) | 답변 (Output) | 평가 |
| :--- | :--- | :--- |
| **안녕?** | 안녕하세요 . | **매우 우수 (정석 답변)** |
| **오늘 기분이 어때?** | 모두 맞춰주가 있었나봐요 . | 보통 (감정 맥락 파악) |
| **배고프다** | 여름 언제나 일기예대한 두려움했겠어요 . | 미흡 (데이터셋 편향 영향) |
| **너는 누구니?** | 하나씩 잊어가는 그 자체만으로 나을 수도 있어요 . | 보통 (철학적 답변) |
| **고민이 있어** | (스타일에) 잠 못 드는 밤인가봐요 . | **우수 (상담 맥락 파악)** |

---

## 4. 프로젝트 회고

### 성과 및 강점
1. **자연스러운 어미 처리:** 초기 학습 시 "그 사람 그 사람"과 같이 단어가 반복되거나 동사 활용이 어색했으나, **에폭을 100까지**하고 **Beam Search**를 도입해서 "안녕하세요", "인가봐요" 등 한국어 특유의 종결 어미를 자연스럽게 생성할 수 있었다.
2. **맥락 파악 능력:** "고민이 있어"라는 입력에 대해 "잠 못 드는 밤인가봐요"라고 답하는 등, 단순히 단어를 매칭하는 수준을 넘어 문장의 정서적 맥락을 파악하는 성능을 보였다.
3. **PyTorch 구현 역량:** 텐서플로우 위주의 예제를 파이토치로 재구현하며 트랜스포머의 동작 원리와 마스킹 메커니즘을 이해했다.

### 한계 & 향후 과제
1. **데이터셋 편향:** 상담 중심의 데이터셋 특성상 "배고프다" 같은 일상적인 질문에도 다소 감상적이거나 엉뚱한 답변이 나오는 경향이 있었다. 더 다양한 일상 대화 데이터를 추가한다면 범용성이 높아질 것이다.
2. **문법적 완결성:** 일부 문장에서 조사가 어색하게 붙는 경우가 있었는데, 데이터 증강(Data Augmentation)이나 빔 서치의 정교한 튜닝을 통해 개선할 수 있을 것으로 판단한다.

### 결론
한국어 전처리부터 트랜스포머 모델 구현, 그리고 고급 디코딩 기법 적용까지 전 과정을 성공적으로 수행했다. 챗봇이 인사에 대해 정확히 응답하고 사용자의 감정 상태를 위로하는 맥락을 성공적으로 형성하면서 프로젝트를 마친다.